In [1]:
import pandas as pd
import numpy as np
import pickle as pkl

In [2]:
from matplotlib import pyplot as plt
import seaborn as sns

In [3]:
from sklearn.metrics import balanced_accuracy_score, accuracy_score

In [4]:
trials = pd.read_parquet(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/trials.pqt"
)

In [5]:
trials_q = trials[trials["mask"]]

In [6]:
LGd_predictions = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/LGd_all_predictions_real.npy"
)
MG_predictions = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/MG_all_predictions_real.npy"
)

In [7]:
def add_congruence(trials):
    out = np.nan_to_num(trials.contrastLeft) - np.nan_to_num(trials.contrastRight)
    left_stim = out > 0
    right_stim = out < 0
    # we throw out 0 contrast trials
    is_left_block = trials["probabilityLeft"] > 0.5  # covers 0.8 blocks
    is_right_block = trials["probabilityLeft"] < 0.5  # covers 0.2 blocks
    congruent = (left_stim & is_left_block) | (right_stim & is_right_block)
    return congruent.values

In [8]:
cx = add_congruence(trials_q)

In [9]:
targets = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_real.npy"
)

In [10]:
targets.shape

(333, 1)

In [11]:
def compute_performance(y_true, y_pred, congruent):
    y_true_cong = y_true[congruent]
    y_true_incong = y_true[~congruent]

    y_pred_cong = y_pred[congruent]
    y_pred_incong = y_pred[~congruent]

    acc_cong = balanced_accuracy_score(y_true_cong, y_pred_cong)
    acc_incong = balanced_accuracy_score(y_true_incong, y_pred_incong)

    # print(f"Congruent Accuracy:   {acc_cong:.3f}")
    # print(f"Incongruent Accuracy: {acc_incong:.3f}")

    return acc_cong, acc_incong

In [12]:
q = compute_performance(targets[:, 0], MG_predictions[0, :], cx)

In [13]:
compute_performance(targets[:, 0], MG_predictions[1, :], cx)

(0.4777154176295807, 0.545)

In [14]:
# i have to save the pseduosessions congruence as well

In [15]:
targets_pseudo = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/targets_pseudo.npy"
)

In [16]:
targets_pseudo.shape

(100, 333, 1)

In [17]:
pseudo_preds = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/MG_all_predictions_pseudo.npy"
)

In [18]:
pseudo_congruence = np.load(
    "../data/ephys_feedback/NYU-11/6713a4a7-faed-4df2-acab-ee4e63326f8d/congruence_pseudo.npy"
)

In [19]:
re = []
for idx in range(100):
    a, b = compute_performance(
        targets_pseudo[idx, :], pseudo_preds[idx, 0, :], pseudo_congruence[0, :]
    )
    re.append([a, b])

In [20]:
re = np.asarray(re)

In [21]:
np.median(re, axis=0)

array([0.49358077, 0.48117844])

In [22]:
# this works

In [24]:
import tempfile
import pickle
import sys
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint

from one.api import ONE
from scipy.special import logit, softmax
import os
from os.path import join
import pickle as pkl
from brainwidemap.bwm_loading import bwm_query, bwm_units, load_trials_and_mask, merge_probes

import concurrent.futures

In [25]:
from manifold.decoding.functions.utils import check_config_decoding

config = check_config_decoding()
MY_REGIONS = config["prior_regions"]
MIN_NEURONS = config["min_units"]

In [26]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)
print("Querying BWM Units...")

units_df = bwm_units(one)
flattened_regions = [item for sublist in MY_REGIONS for item in sublist]
relevant_pids = units_df[units_df["Beryl"].isin(flattened_regions)]["pid"].unique()

bwm_df = bwm_query(one)
# change subset df: use all valid eids
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]
list_of_eids = subset_df["eid"].unique()

Querying BWM Units...
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [28]:
# check if the segmentation is unique

In [29]:
config = check_config_decoding()
MY_REGIONS = config["prior_regions"]
MIN_NEURONS = config["min_units"]

In [31]:
truncated_eids = [eid.split("-")[0] for eid in list_of_eids]

In [33]:
len(np.unique(truncated_eids)) == len(list_of_eids)

True